# Prepare data

## 0. Setup

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
ORIGIN_DIR = PROJECT_DIR / "0_origin"
PROCESSED_DIR = PROJECT_DIR / "1_processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Station data

In [14]:
df_stations = pd.read_csv(ORIGIN_DIR / "Station_registry_information.CSV", header=0, index_col="AirQualityStation")
print(f"Total number of stations: {df_stations.shape[0]}; features: {df_stations.shape[1]}")
df_stations.head()

Total number of stations: 744; features: 14


,Longitude,Latitude,Altitude,AirQualityStationType,AirQualityStationArea,COD_RIP,COD_REG,COD_PROV,COD_CM,COD_UTS,PRO_COM,PRO_COM_T,COMUNE,Area comune
AirQualityStation,,,,,,,,,,,,,,
STA.IT0063A,10.534167,42.939167,37.0,industrial,suburban,3,9,49,0,49,49012,49012,Piombino,1.307445e+08
STA.IT0187A,11.613060,44.842500,8.0,traffic,urban,2,8,38,0,38,38008,38008,Ferrara,4.055220e+08
STA.IT0267A,9.414722,45.329444,80.0,background,suburban,1,3,98,0,98,98056,98056,Tavazzano con Villavesco,1.606860e+07
STA.IT0448A,12.313889,45.429444,1.0,background,urban,2,5,27,227,227,27042,27042,Venezia,4.183949e+08
STA.IT0459A,13.341940,43.598890,15.0,background,suburban,3,11,42,0,42,42014,42014,Chiaravalle,1.760196e+07


Add province, region and area name

In [15]:
df_stations = df_stations.rename(columns={"COMUNE": "Comune"})

df_comuni = pd.read_excel(
    ORIGIN_DIR / "Elenco-comuni-italiani.xlsx",
    sheet_name=0,
    usecols=[5, 9, 10, 11],
)
df_comuni.columns = [
    "Comune",
    "Ripartizione geografica",
    "Regione",
    "Provincia",
]
df_comuni = df_comuni.loc[df_comuni["Comune"].isin(df_stations["Comune"])]

df_stations = df_stations.join(
    df_comuni.set_index("Comune"),
    on="Comune",
    how="left",
    validate="many_to_one",
)
print(
    "Stations without a municipality match: "
    f"{df_stations['Ripartizione geografica'].isna().sum()}"
)

Stations without a municipality match: 0


Remove useless columns and save

In [17]:
df_stations = df_stations[["AirQualityStationType", "AirQualityStationArea",
                          "Comune", "Provincia", "Regione", "Ripartizione geografica",
                          "Longitude", "Latitude", "Altitude"]].copy()
df_stations.to_csv(PROCESSED_DIR / "station_info.csv")
df_stations.head()


,AirQualityStationType,AirQualityStationArea,Comune,Provincia,Regione,Ripartizione geografica,Longitude,Latitude,Altitude
AirQualityStation,,,,,,,,,
STA.IT0063A,industrial,suburban,Piombino,Livorno,Toscana,Centro,10.534167,42.939167,37.0
STA.IT0187A,traffic,urban,Ferrara,Ferrara,Emilia-Romagna,Nord-est,11.613060,44.842500,8.0
STA.IT0267A,background,suburban,Tavazzano con Villavesco,Lodi,Lombardia,Nord-ovest,9.414722,45.329444,80.0
STA.IT0448A,background,urban,Venezia,Venezia,Veneto,Nord-est,12.313889,45.429444,1.0
STA.IT0459A,background,suburban,Chiaravalle,Ancona,Marche,Centro,13.341940,43.598890,15.0
